**CLASS - 27-01-2026**

*01. ENCODER - DECODER ONLY MODEL FOR LANGUAGE TRANSLATION*

In [ ]:

import tensorflow as tf
import numpy as np
import re
import os
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



# 2. LOAD DATASET (ENGLISH → FRENCH)

path = tf.keras.utils.get_file(
    "fra-eng.zip",
    "https://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
    extract=True
)


text_path = os.path.join(path, "fra.txt")



# 3. PREPROCESSING

def clean(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"[^a-zA-Z?.!,]+", " ", sentence)
    return sentence.strip()

pairs = []

with open(text_path, encoding="utf-8") as f:
    for line in f.readlines()[:10000]:
        eng, fra = line.split("\t")[:2]
        pairs.append((clean(eng), clean(fra)))



# 4. TOKENIZATION

# Encoder (English)
src_tokenizer = Tokenizer(filters="")
src_tokenizer.fit_on_texts([p[0] for p in pairs])

encoder_input = pad_sequences(
    src_tokenizer.texts_to_sequences([p[0] for p in pairs]),
    padding="post"
)

# Decoder (French)
tgt_tokenizer = Tokenizer(filters="")
tgt_tokenizer.fit_on_texts(
    ["<sos> " + p[1] + " <eos>" for p in pairs]
)

decoder_input = pad_sequences(
    tgt_tokenizer.texts_to_sequences(
        ["<sos> " + p[1] for p in pairs]
    ),
    padding="post"
)

decoder_output = pad_sequences(
    tgt_tokenizer.texts_to_sequences(
        [p[1] + " <eos>" for p in pairs]
    ),
    padding="post"
)

src_vocab_size = len(src_tokenizer.word_index) + 1
tgt_vocab_size = len(tgt_tokenizer.word_index) + 1

sos_token = tgt_tokenizer.word_index["<sos>"]
eos_token = tgt_tokenizer.word_index["<eos>"]
max_len = decoder_input.shape[1]



# 5. MODEL (ENCODER – DECODER ONLY)

class Encoder(Model):
    def __init__(self, vocab_size, embedding_dim, hidden_units):
        super().__init__()
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(hidden_units, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        _, h, c = self.lstm(x)
        return h, c


class Decoder(Model):
    def __init__(self, vocab_size, embedding_dim, hidden_units):
        super().__init__()
        self.embedding = Embedding(vocab_size, embedding_dim)
        self.lstm = LSTM(hidden_units, return_sequences=True, return_state=True)
        self.fc = Dense(vocab_size, activation="softmax")

    def call(self, x, states):
        x = self.embedding(x)
        output, _, _ = self.lstm(x, initial_state=states)
        return self.fc(output)


class Seq2Seq(Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        enc_inp, dec_inp = inputs
        states = self.encoder(enc_inp)
        return self.decoder(dec_inp, states)


encoder = Encoder(src_vocab_size, 128, 256)
decoder = Decoder(tgt_vocab_size, 128, 256)
model = Seq2Seq(encoder, decoder)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)



# 6. TRAINING

model.fit(
    [encoder_input, decoder_input],
    decoder_output,
    epochs=15,
    batch_size=64
)



# 7. TRANSLATION FUNCTION (INFERENCE)

index_to_word = {v: k for k, v in tgt_tokenizer.word_index.items()}

def translate(sentence):
    sentence = clean(sentence)
    seq = src_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=encoder_input.shape[1], padding="post")

    states = encoder(seq)
    target = tf.constant([[sos_token]])

    result = []

    for _ in range(max_len):
        output = decoder(target, states)
        word_id = tf.argmax(output[0, -1]).numpy()

        if word_id == eos_token:
            break

        result.append(index_to_word[word_id])
        target = tf.constant([[word_id]])

    return " ".join(result)



# 8. TEST TRANSLATION

print("EN:", "i am a student")
print("FR:", translate("i am a student"))

print("EN:", "how are you")
print("FR:", translate("how are you"))

print("EN:", "i love machine learning")
print("FR:", translate("i love machine learning"))

Epoch 1/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.6227 - loss: 3.7572
Epoch 2/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.7201 - loss: 1.7539
Epoch 3/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.7383 - loss: 1.5589
Epoch 4/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.7690 - loss: 1.3940
Epoch 5/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.7843 - loss: 1.2642
Epoch 6/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.7989 - loss: 1.1619
Epoch 7/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.8107 - loss: 1.0697
Epoch 8/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.8188 - loss: 0.9965
Epoch 9/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.8244 - loss: 0.9292
Epoch 10/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - accuracy: 0.8322 - loss: 0.8637
Epoch 11/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.8392 - loss: 0.7967
Epoch 12/15
157/157 ━━━━━━━━━━━━━━━━━━━━ 

*02.Using API's*

In [ ]:
pip install requests


In [ ]:
import requests

def translate_text(text, target_lang):
    url = "https://api.mymemory.translated.net/get"
    params = {
        "q": text,
        "langpair": f"en|{target_lang}"
    }

    response = requests.get(url, params=params)
    data = response.json()

    return data["responseData"]["translatedText"]



print("English : I am shyam..beat me if you can")
print("Tamil   :", translate_text("I am shyam..beat me if you can", "ta"))

print("\nEnglish : How are you")
print("French  :", translate_text("How are you", "fr"))

print("\nEnglish : I love machine learning")
print("Hindi   :", translate_text("I love machine learning", "hi"))


English : I am shyam..beat me if you can
Tamil   : நான் ஷ்யாம்.. உங்களால் முடிந்தால் என்னை அடிக்கவும்

English : How are you
French  : Comment allez-vous

English : I love machine learning
Hindi   : मुझे मशीन लर्निंग पसंद है
